# 03 — Train Neural Collaborative Filtering (NCF) on Kaggle GPU

Trains the NeuMF model on MovieLens 25M. Save + commit the notebook to persist `ncf.pth` in the output.

**Prerequisites:**
- Runtime: **GPU T4 × 2** (on a 2-GPU session) or P100
- Internet: **ON** (Settings → Internet toggle) — needed to clone repo + download data
- Data: either let the notebook download MovieLens, or add `grouplens/movielens-25m` as a Dataset input

**Architecture:** NeuMF = GMF (linear) + MLP (nonlinear) fused via sigmoid. Paper: https://arxiv.org/abs/1708.05031

**Output:** `/kaggle/working/ncf.pth` → commit the notebook and download from output tab.

## 1. Setup

In [ ]:
import os, sys, time, urllib.request, zipfile
import numpy as np, pandas as pd, torch
from torch.utils.data import Dataset, DataLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpu_count = torch.cuda.device_count()
print(f'torch {torch.__version__}  device={device}  GPUs={gpu_count}')
for g in range(gpu_count):
    print(f'  GPU {g}: {torch.cuda.get_device_name(g)}')

## 2. Clone the repo

Imports the exact model code from the repo so local `make evaluate` loads the same architecture.

In [ ]:
if not os.path.isdir('recsys-engine'):
    !git clone https://github.com/its-Sohan/recsys-engine.git
sys.path.insert(0, 'recsys-engine')
from src.models.ncf import NeuMF
print('NeuMF imported')

## 3. Download MovieLens 25M

If you added `grouplens/movielens-25m` as a Dataset input, set `DATA_DIR = '/kaggle/input/movielens-25m'` and skip this cell. Otherwise, internet download (~250MB).

In [ ]:
DATA_DIR = 'ml-25m'
if not os.path.isdir(DATA_DIR):
    url = 'https://files.grouplens.org/datasets/movielens/ml-25m.zip'
    print('Downloading...')
    urllib.request.urlretrieve(url, 'ml-25m.zip')
    with zipfile.ZipFile('ml-25m.zip') as z:
        z.extractall('.')
    os.remove('ml-25m.zip')
print('Dataset ready:', os.listdir(DATA_DIR))

## 4. Load + time-based split

In [ ]:
ratings = pd.read_csv(f'{DATA_DIR}/ratings.csv',
    dtype={'userId': np.int32, 'movieId': np.int32, 'rating': np.float32, 'timestamp': np.int64})
ratings = ratings.sort_values('timestamp').reset_index(drop=True)
n_test = int(len(ratings) * 0.2)
train = ratings.iloc[:-n_test].copy()
test  = ratings.iloc[-n_test:].copy()
print(f'train: {len(train):,}  test: {len(test):,}')

## 5. Streaming negative-sampling dataset

Generates negatives on-the-fly instead of precomputing 125M tuples (~3GB RAM saved). Standard for large-scale NCF.

In [ ]:
users = np.sort(train.userId.unique())
items = np.sort(train.movieId.unique())
user2idx = {u: i for i, u in enumerate(users)}
item2idx = {it: i for i, it in enumerate(items)}
idx2item = {i: it for it, i in item2idx.items()}
print(f'{len(user2idx):,} users, {len(item2idx):,} items')


class StreamingNCFDataset(Dataset):
    def __init__(self, positives, user2idx, item2idx, n_items, neg_ratio=4, seed=42):
        self.pos_users = positives['userId'].map(user2idx).to_numpy(dtype=np.int64)
        self.pos_items = positives['movieId'].map(item2idx).to_numpy(dtype=np.int64)
        self.n_pos = len(self.pos_users)
        self.neg_ratio = neg_ratio
        self.n_items = n_items
        self.rng = np.random.default_rng(seed)
        seen = positives.groupby('userId')['movieId'].apply(set).to_dict()
        self.user_seen_idx = {
            user2idx[u]: {item2idx[i] for i in seen.get(u, set()) if i in item2idx}
            for u in user2idx
        }

    def __len__(self):
        return self.n_pos * (1 + self.neg_ratio)

    def __getitem__(self, idx):
        if idx < self.n_pos:
            u = int(self.pos_users[idx])
            i = int(self.pos_items[idx])
            label = 1
        else:
            pos_idx = idx // (1 + self.neg_ratio)
            u = int(self.pos_users[pos_idx])
            seen_set = self.user_seen_idx.get(u, set())
            i = int(self.rng.integers(0, self.n_items))
            for _ in range(5):
                if i not in seen_set:
                    break
                i = int(self.rng.integers(0, self.n_items))
            label = 0
        return np.int64(u), np.int64(i), np.float32(label)


positives = train[train['rating'] >= 4.0]
dataset = StreamingNCFDataset(positives, user2idx, item2idx, n_items=len(item2idx))
loader = DataLoader(dataset, batch_size=1024 * max(1, gpu_count),
                    shuffle=True, num_workers=2)
print(f'training tuples: {len(dataset):,}  (streamed, not precomputed)')

## 5.5 — DataParallel (2× GPU)

If 2 GPUs detected, wraps model so the batch is split across both automatically.

In [ ]:
model = NeuMF(len(user2idx), len(item2idx))
if gpu_count > 1:
    model = torch.nn.DataParallel(model, device_ids=list(range(gpu_count)))
    print(f'DataParallel on {gpu_count} GPUs')
model = model.to(device)

## 6. Train

~10–15 min for 15 epochs on 2× T4. Loss: 0.693 (random) → ~0.30–0.40.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.BCELoss()

EPOCHS = 15
model.train()
for epoch in range(EPOCHS):
    t0 = time.time()
    total, n = 0.0, 0
    for bu, bi, bl in loader:
        bu, bi, bl = bu.to(device), bi.to(device), bl.to(device)
        optimizer.zero_grad()
        preds = model(bu, bi)
        loss = criterion(preds, bl)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(bu)
        n += len(bu)
    print(f'epoch {epoch+1:2d}/{EPOCHS}  loss={total/n:.4f}  ({time.time()-t0:.0f}s)')

## 7. Save checkpoint

Unwraps DataParallel if needed. Saves to `/kaggle/working/ncf.pth` — commit the notebook to persist it, then download from the Output tab.

In [ ]:
if hasattr(model, 'module'):
    state_dict = model.module.state_dict()
else:
    state_dict = model.state_dict()

seen = train.groupby('userId')['movieId'].apply(set).to_dict()
payload = {
    'state_dict': state_dict,
    'config': {'gmf_dim': 64, 'mlp_embed_dim': 32, 'mlp_layers': [64, 32, 16, 8]},
    'user2idx': user2idx,
    'idx2item': {int(k): int(v) for k, v in idx2item.items()},
    'seen': {int(k): list(v) for k, v in seen.items()},
}
save_path = '/kaggle/working/ncf.pth'
torch.save(payload, save_path)
size_mb = os.path.getsize(save_path) / 1e6
print(f'Saved ncf.pth ({size_mb:.1f} MB) -> {save_path}')
print('Commit notebook to persist, then download from Output tab.')

# Also copy to working dir root so it appears in commit content
import shutil
shutil.copy(save_path, 'ncf.pth')

## 8. Next steps

After downloading `ncf.pth`, place it at `artifacts/ncf.pth` in your local repo:

```
cp ~/Downloads/ncf.pth /home/sohan/projects/recsys-engine/artifacts/
cd /home/sohan/projects/recsys-engine
python -m src.models.train --load-ncf artifacts/ncf.pth
make evaluate
```

This loads the checkpoint, converts to `.joblib`, and runs evaluation against the same time-based test split.

## What just happened

1. **GMF** learned linear user-item interactions (generalized dot product).
2. **MLP** learned nonlinear interactions via stacked dense layers.
3. **NeuMF fusion** weights both branches via sigmoid.
4. **Streaming negatives** avoided 3GB RAM.
5. **DataParallel** split across 2 GPUs.